# Contrast Enhancement Walkthrough: CLAHE & Retinex

End-to-end demo of the enhancement pipelines from `src/enhancement.py`.
Uses a synthetic low-contrast test image so the notebook runs anywhere —
replace `load_image()` with your own diagnostic images to run on real data.

In [ ]:
import sys
sys.path.append("../src")
import numpy as np
import cv2
import matplotlib.pyplot as plt
from enhancement import compare_methods

## 1. Load (or synthesize) a low-contrast image

In [ ]:
def load_image(path: str | None = None) -> np.ndarray:
    """Load a grayscale image, or synthesize a low-contrast test image."""
    if path:
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(path)
        return img
    # Synthetic low-contrast phantom: soft gradient + shapes + noise
    rng = np.random.default_rng(42)
    y, x = np.mgrid[0:256, 0:256]
    base = 90 + 25 * np.sin(x / 30) * np.cos(y / 40)
    cv2.circle(base, (128, 128), 60, 130, -1)
    cv2.rectangle(base, (40, 40), (90, 90), 115, -1)
    noisy = base + rng.normal(0, 6, base.shape)
    return np.clip(noisy, 0, 255).astype(np.uint8)

img = load_image()  # pass a path to use a real image
print("shape:", img.shape, "dtype:", img.dtype, "range:", img.min(), "-", img.max())

## 2. Run all enhancement methods

In [ ]:
results = compare_methods(img)
print("methods:", list(results.keys()))

## 3. Visual comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, out) in zip(axes, results.items()):
    ax.imshow(out, cmap="gray", vmin=0, vmax=255)
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Quantitative check: contrast gain

Simple proxy metric — standard deviation of pixel intensities (higher ≈ more contrast).
TODO: extend with EME / AMBE metrics for a proper benchmark.

In [ ]:
for name, out in results.items():
    print(f"{name:10s} std={out.std():7.2f}  mean={out.mean():7.2f}")